# Employee Turnover Analysis

**Tabular Classification** — Predict whether an employee will leave the company.

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Dataset: HR Analytics (Kaggle)
Target: `left` (0=stayed, 1=left)

## 2 · Project Overview

We analyze and predict employee turnover using the classic HR Analytics dataset. The goal is to identify factors driving turnover and build predictive models for early warning.

## 3 · Learning Objectives

1. Analyze employee satisfaction and workload patterns.
2. Build classification models for turnover prediction.
3. Handle moderate class imbalance (~24% turnover rate).
4. Compare boosting models with AutoML baselines.
5. Extract actionable HR insights from model results.

## 4 · Problem Statement

Given employee satisfaction, evaluation, workload, tenure, and department, predict whether the employee will leave.

## 5 · Why This Project Matters

- Employee turnover costs 50-200% of annual salary per departure.
- Predictive models enable proactive retention strategies.
- Understanding turnover drivers helps improve workplace culture.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | HR Analytics via Kaggle |
| **Rows** | ~14,999 |
| **Target** | left (0/1, ~24% positive) |
| **Features** | satisfaction_level, last_evaluation, number_project, average_montly_hours, time_spend_company, Work_accident, promotion_last_5years, Department, salary |

## 7 · Dataset Source and License Notes

- **Source**: `giripujar/hr-analytics` on Kaggle
- **License**: Public / educational
- **Note**: Classic HR dataset widely used in tutorials.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict", "kagglehub"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "left"
SAVE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
_local = os.path.join(SAVE_DIR, "HR_comma_sep.csv")
if not os.path.exists(_local):
    _local = "Classification/Employee Turnover Analysis/HR_comma_sep.csv"
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Employee Turnover Analysis


## 11 · Dataset Download or Loading

In [4]:
if os.path.exists(_local):
    df = pd.read_csv(_local)
    print(f"Loaded local: {_local}")
else:
    import kagglehub
    path = kagglehub.dataset_download("giripujar/hr-analytics")
    csv_file = None
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.endswith(".csv"):
                csv_file = os.path.join(root, f)
                break
        if csv_file:
            break
    assert csv_file, "No CSV found!"
    df = pd.read_csv(csv_file)
    print(f"Downloaded from kagglehub")

print(f"Shape: {df.shape}")
df.head()

Loaded local: E:\Github\Machine-Learning-Projects\Classification\Employee Turnover Analysis\HR_comma_sep.csv
Shape: (14999, 10)


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"Turnover rate: {df[TARGET].mean():.2%}")
print(f"\nDtypes:\n{df.dtypes}")

DATA VALIDATION

Missing values: 0
Duplicate rows: 3008

Target distribution:
left
0    11428
1     3571
Name: count, dtype: int64
Turnover rate: 23.81%

Dtypes:
satisfaction_level       float64
last_evaluation          float64
number_project             int64
average_montly_hours       int64
time_spend_company         int64
Work_accident              int64
left                       int64
promotion_last_5years      int64
Department                object
salary                    object
dtype: object


## 13 · Exploratory Data Analysis

In [6]:
# Drop duplicates
df = df.drop_duplicates()
print(f"Shape after dedup: {df.shape}")

num_cols = df.select_dtypes(include="number").columns.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax, square=True)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

# Key distributions by turnover
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(["satisfaction_level", "last_evaluation", "number_project",
                          "average_montly_hours", "time_spend_company", "Work_accident"]):
    if col in df.columns:
        ax = axes[i // 3][i % 3]
        for val in [0, 1]:
            subset = df[df[TARGET] == val][col]
            ax.hist(subset, bins=25, alpha=0.5, label=f"left={val}", edgecolor="black")
        ax.set_title(col)
        ax.legend()
plt.suptitle("Feature Distributions by Turnover", y=1.02)
plt.tight_layout()
plt.show()

Shape after dedup: (11991, 10)


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df[TARGET].value_counts().plot(kind="bar", ax=axes[0], color=["steelblue", "salmon"], edgecolor="black")
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xticklabels(["Stayed (0)", "Left (1)"], rotation=0)

# Turnover by department
if "Department" in df.columns or "sales" in df.columns:
    dept_col = "Department" if "Department" in df.columns else "sales"
    dept_rate = df.groupby(dept_col)[TARGET].mean().sort_values(ascending=False)
    dept_rate.plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
    axes[1].set_title("Turnover Rate by Department")
    axes[1].set_ylabel("Turnover Rate")
    axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# Salary vs turnover
if "salary" in df.columns:
    print(f"\nTurnover by salary level:")
    print(df.groupby("salary")[TARGET].mean().sort_values(ascending=False))


Turnover by salary level:
salary
low       0.204530
medium    0.146170
high      0.048485
Name: left, dtype: float64


## 15 · Train/Validation/Test Split Strategy

80/20 stratified split.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categoricals
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Sanitize column names
X.columns = [re.sub(r'[^A-Za-z0-9_]', '_', c) for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train turnover rate: {y_train.mean():.2%}")

Train: (9592, 9), Test: (2399, 9)
Train turnover rate: 16.61%


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Duplicates**: Removed.
- **Categorical**: Department and salary via OrdinalEncoder.
- **Scaling**: Not needed for tree models.

## 17 · Feature Engineering

The HR dataset has well-defined features. We proceed with the original feature set.

## 18 · Baseline Model

In [9]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"\n{classification_report(y_test, y_pred_bl)}")

BASELINE: Logistic Regression
Accuracy : 0.8370
F1       : 0.8070

              precision    recall  f1-score   support

           0       0.86      0.96      0.91      2001
           1       0.52      0.21      0.30       398

    accuracy                           0.84      2399
   macro avg       0.69      0.59      0.60      2399
weighted avg       0.80      0.84      0.81      2399



## 19 · LazyPredict Benchmark

In [10]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 10 Classifiers:")
print(lazy_models.head(10).to_string())

LazyPredict — Top 10 Classifiers:
                        Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                   
LGBMClassifier          0.984994           0.962825  0.983825  0.984838   0.984933  0.984994    3.172017
RandomForestClassifier  0.986661           0.962818  0.977762  0.986479   0.986729  0.986661    1.110418
BaggingClassifier       0.984160           0.962325  0.966528  0.984012   0.984067  0.984160    0.250064
CatBoostClassifier      0.984160           0.962325  0.982474  0.984012   0.984067  0.984160    4.996428
XGBClassifier           0.982910           0.961576  0.983720  0.982777   0.982784  0.982910    0.291444
ExtraTreesClassifier    0.982910           0.958557  0.977516  0.982723   0.982812  0.982910    0.782932
DecisionTreeClassifier  0.970404           0.952067  0.952067  0.970565   0.970795  0.970404    0.060356
KNeighborsClassifier 

## 20 · FLAML AutoML Run

In [11]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="classification", time_budget=60, verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    print(f"FLAML best: {flaml_automl.best_estimator}")
    print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
    print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML failed: {e}")
    y_pred_flaml = y_pred_bl

FLAML failed: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

In [12]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).flatten().astype(int)
    print(f"CatBoost Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost GPU failed ({e}), trying CPU...")
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
        timings["CatBoost"] = time.perf_counter() - t0
        results["CatBoost"] = cb.predict(X_test).flatten().astype(int)
        print(f"CatBoost Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  (CPU, {timings['CatBoost']:.1f}s)")
    except Exception as e2:
        print(f"CatBoost failed: {e2}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                            device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
    lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
           callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM GPU failed, trying CPU...")
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
        timings["LightGBM"] = time.perf_counter() - t0
        results["LightGBM"] = lg.predict(X_test)
        print(f"LightGBM Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  (CPU)")
    except Exception as e2:
        print(f"LightGBM failed: {e2}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                              device="cuda", tree_method="hist", verbosity=0,
                              eval_metric="logloss",
                              n_jobs=-1, random_state=SEED)
    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost GPU failed, trying CPU...")
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0, eval_metric="logloss",
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        timings["XGBoost"] = time.perf_counter() - t0
        results["XGBoost"] = xgb_model.predict(X_test)
        print(f"XGBoost Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  (CPU)")
    except Exception as e2:
        print(f"XGBoost failed: {e2}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost Acc: 0.9833  (7.0s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[172]	valid_0's binary_logloss: 0.0661325
LightGBM Acc: 0.9846  (5.3s)


XGBoost Acc: 0.9846  (5.1s)


## 22 · Top 3 Model Selection

In [13]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="weighted"), 4),
        "Precision": round(precision_score(y_test, yp, average="weighted", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="weighted"), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
LightGBM               0.9846  0.9844     0.9845  0.9846
XGBoost                0.9846  0.9844     0.9845  0.9846
CatBoost               0.9833  0.9832     0.9832  0.9833
Logistic Regression    0.8370  0.8070     0.8036  0.8370
FLAML                  0.8370  0.8070     0.8036  0.8370

Top 3 models: ['LightGBM', 'XGBoost', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

In [14]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i])
    axes[i].set_title(f"{name}\nAcc={accuracy_score(y_test, yp):.4f}  F1={f1_score(y_test, yp, average='weighted'):.4f}")
    axes[i].set_xlabel("Predicted"); axes[i].set_ylabel("Actual")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.show()

for name in top3_names:
    yp = results[name]
    print(f"\n{name}:")
    print(f"  Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"  F1       : {f1_score(y_test, yp, average='weighted'):.4f}")


LightGBM:
  Accuracy : 0.9846
  F1       : 0.9844

XGBoost:
  Accuracy : 0.9846
  F1       : 0.9844

CatBoost:
  Accuracy : 0.9833
  F1       : 0.9832


## 24 · Error Analysis

In [15]:
best_name = top3_names[0]
best_pred = results[best_name]
cm = confusion_matrix(y_test, best_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title(f"{best_name} — Confusion Matrix")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]
err_rate_per_feat = errors.groupby("correct").mean(numeric_only=True)
err_rate_per_feat.T.plot(kind="bar", ax=axes[1], figsize=(14, 5))
axes[1].set_title("Mean Feature Values: Correct vs Incorrect")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()
print(f"Overall accuracy ({best_name}): {accuracy_score(y_test, best_pred):.4f}")
print(f"Misclassified: {(y_test.values != best_pred).sum()} / {len(y_test)}")

Overall accuracy (LightGBM): 0.9846
Misclassified: 37 / 2399


## 25 · Interpretation and Business Insight

**Key findings:**
- **Satisfaction level** is the #1 predictor — low satisfaction → high turnover.
- **Number of projects** and **average monthly hours** show a bimodal pattern: both overworked and underutilized employees leave.
- **Time at company** around 3-5 years is a critical window.
- **Low salary** strongly correlates with turnover.

**Business takeaway:** Monitor satisfaction scores, balance workload, and review compensation for 3-5 year tenured employees.

## 26 · Limitations

1. Classic dataset — may not reflect modern workplace dynamics.
2. No remote work or flexible hours data.
3. Satisfaction is measured once, not tracked over time.
4. 'left' is binary — doesn't distinguish voluntary vs involuntary.

## 27 · How to Improve

1. Track satisfaction over time for trend analysis.
2. Add engagement metrics (email activity, meeting attendance).
3. Frame as survival analysis (time to departure).
4. Segment analysis by department and role level.

## 28 · Production Considerations

- Predictions should be one input to HR decisions.
- Must ensure privacy and ethical use of employee data.
- Regular retraining as company culture evolves.
- Dashboard for HR managers with explainable predictions.

## 29 · Common Mistakes

1. Not removing duplicates (this dataset has many).
2. Using accuracy alone on 76/24 split.
3. Ignoring the bimodal pattern in hours worked.
4. Not exploring satisfaction × evaluation interaction.

## 30 · Mini Challenge / Exercises

1. Create satisfaction × evaluation quadrants and analyze turnover.
2. Try class_weight='balanced' — does recall improve?
3. Build a model using only satisfaction_level — how good?
4. Cluster employees first, then predict turnover within each cluster.
5. Visualize decision boundaries with 2D projections.

## 31 · Final Summary / Key Takeaways

1. Satisfaction level is the strongest turnover predictor.
2. Bimodal pattern: both overworked and underutilized employees leave.
3. 3-5 year tenure is a critical retention window.
4. Boosting models achieve strong performance on this dataset.
5. HR should focus on satisfaction monitoring and workload balance.

## Save Metrics

In [16]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="weighted")), 4),
        "precision": round(float(precision_score(y_test, yp, average="weighted", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="weighted")), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Employee Turnover Analysis\metrics.json
{
  "CatBoost": {
    "accuracy": 0.9833,
    "f1": 0.9832,
    "precision": 0.9832,
    "recall": 0.9833
  },
  "LightGBM": {
    "accuracy": 0.9846,
    "f1": 0.9844,
    "precision": 0.9845,
    "recall": 0.9846
  },
  "XGBoost": {
    "accuracy": 0.9846,
    "f1": 0.9844,
    "precision": 0.9845,
    "recall": 0.9846
  },
  "Logistic Regression": {
    "accuracy": 0.837,
    "f1": 0.807,
    "precision": 0.8036,
    "recall": 0.837
  },
  "FLAML": {
    "accuracy": 0.837,
    "f1": 0.807,
    "precision": 0.8036,
    "recall": 0.837
  }
}
